In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.1 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import csv

DATASET_ROOT = "/content/crcsegformer-2"
OUTPUT_ROOT = "/content/dataset_yolo"

SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}

os.makedirs(OUTPUT_ROOT, exist_ok=True)
classes = None

for split, out_split in SPLIT_MAP.items():
    src_dir = os.path.join(DATASET_ROOT, split)
    if not os.path.isdir(src_dir):
        continue

    img_out = os.path.join(OUTPUT_ROOT, "images", out_split)
    mask_out = os.path.join(OUTPUT_ROOT, "masks", out_split)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)

    for fname in os.listdir(src_dir):
        fpath = os.path.join(src_dir, fname)

        if fname == "_classes.csv":
            if classes is None:
                with open(fpath, newline="") as f:
                    reader = csv.reader(f)
                    rows = [r for r in reader if r]
                    # skip header if present
                    if rows and not rows[0][0].strip().isdigit():
                        rows = rows[1:]
                    classes = {int(r[0].strip()): r[1].strip() for r in rows}
            continue

        if fname.endswith("_mask.png"):
            stem = fname[: -len("_mask.png")]
            shutil.copy(fpath, os.path.join(mask_out, stem + ".png"))
        else:
            shutil.copy(fpath, os.path.join(img_out, fname))

    print(f"{split}: done")

if classes is None:
    raise RuntimeError("No _classes.csv found — check DATASET_ROOT path")

yaml_path = os.path.join(OUTPUT_ROOT, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {OUTPUT_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("masks_dir: masks\n\n")
    f.write("names:\n")
    for idx, name in sorted(classes.items()):
        f.write(f"  {idx}: {name}\n")

print(f"\nWrote {yaml_path}")
print("Classes found:", classes)

train: done
valid: done
test: done

Wrote /content/dataset_yolo/data.yaml
Classes found: {0: 'background', 1: 'Crack'}


In [ ]:
import os
print(os.listdir("/content/dataset_yolo/images/train")[:10])

['00043_jpg.rf.c8509d2be70de84e3da7cd3773429343.jpg', 'CFD_044_jpg.rf.f9db1c0301980e8a173e83d01a9d5ea7.jpg', '00090_jpg.rf.d07f5afd68304e0c2b65e18638aa6213.jpg', '00160_jpg.rf.ceef66b0e53e23f1f543313045331907.jpg', '00076_jpg.rf.9a09bdad2d04c3d7f1f48b4fae41eb10.jpg', 'CFD_009_jpg.rf.a89506f8d5f9609fba1504537d20c94b.jpg', 'CFD_022_jpg.rf.63d375ad7811b15481cf552c444459e7.jpg', '00175_jpg.rf.48359acd281f4f0e0ddb853423f65ef3.jpg', 'CFD_019_jpg.rf.49d94bd9682b6fee39374cccdbd7740c.jpg', 'CFD_109_jpg.rf.8c404ca19d4fbcaff127a7291a79e3d3.jpg']


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26x-sem.pt")
results = model.train(data="/content/dataset_yolo/data.yaml", epochs=100, imgsz=1024)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2711/1683509470.py", line 3, in <cell line: 0>
    results = model.train(data="/content/dataset_yolo/data.yaml", epochs=100, imgsz=1024)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 836, in train
    self.trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 243, in train
    self._do_train()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 510, in _do_train
    self.scaler.scale(self.loss).backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.12/dist-packages/

TypeError: object of type 'NoneType' has no len()

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/runs/semantic/train/weights/best.pt")
results = model("/content/istockphoto-2220265384-640_adpp_is.mp4", save=True)
for result in results:
    semantic_mask = result.semantic_mask.data


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/230) /content/istockphoto-2220265384-640_adpp_is.mp4: 576x1024 89.4ms
video 1/1 (frame 2/230) /content/istockphoto-2220265384-640_adpp_is.mp4: 576x1024 58.7ms
video 1/1 (frame 3/230) /content/istockphoto-2220265384-640_adpp_is.mp4: 576x1024 59.5ms
video 1/1 (frame 4/230) /content/istockphoto-2220265384-640_adpp_is.mp4: 576x1024 57.8ms
video 1/1 (frame 5/230) /content/istockphoto-2220265384-640_adpp_is.mp4: 576x1024 57.1ms
video 1/1 (fr